# Specialized multi-agent review
We test the full panel: novelty, feasibility, clarity/importance, and contrarian review.
The agents share a pretrained model but have different task instructions and structured contracts.
The novelty role searches the downloaded ACL corpus locally using BM25. Online retrieval remains optional.


**Corrected live pilot (`smoke_v2`):** All twelve final-condition rows have accepted independent ratings, and all sixteen distinct judged proposals passed validation. One generic-review run stopped early on a wrong-role response. The full panel has higher mean feasibility, clarity and grounding but lower mean novelty than one-shot in this two-seed sample. These are preliminary proposal-level ratings; no statistical superiority or human validation is claimed. The earlier quotation-failure pilot remains preserved in `results/smoke`.


In [1]:
from pathlib import Path
import sys, os, json
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'research_review').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from research_review.config import Settings, budget
from research_review.schemas import Paper, Run
from research_review.storage import load_json
from research_review.literature_retrieval import create_retriever
from research_review.llm import GeminiClient
from research_review.experiments import run_study
settings = Settings.from_env()
study = ROOT / 'results' / os.getenv('RESEARCH_REVIEW_STUDY', 'smoke_v2')
seeds = [Paper.model_validate(p) for p in load_json(ROOT / 'data' / 'smoke_seeds.json')]
retriever = create_retriever(settings)
RUN_LIVE = os.getenv('RUN_LIVE_EXPERIMENTS') == '1'
print('Model:', settings.model, '| rounds:', settings.rounds)
print('Credentials configured:', bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')))
print('Live execution opt-in:', RUN_LIVE)
print('Fixed smoke seeds:', [(p.paper_id, p.year, p.title) for p in seeds])


Model: gemini-3.5-flash-lite | rounds: 3
Credentials configured: True
Live execution opt-in: False
Fixed smoke seeds: [('2025.naacl-long.342', 2025, 'ResearchAgent: Iterative Research Idea Generation over Scientific Literature with Large Language Models'), ('2024.acl-long.18', 2024, 'SciMON: Scientific Inspiration Machines Optimized for Novelty')]


## Roles, evidence and independence
The novelty role compares proposed mechanisms with searched prior work. Feasibility checks data,
compute, implementation and time. Clarity/importance checks definitions, coherence and scientific value.
The contrarian challenges assumptions and proposes falsification tests.

Each reviewer receives a separate prompt containing the current proposal and appropriate literature,
without other critiques. Sequential API requests preserve this information independence and reduce
free-tier rate pressure. A deterministic aggregator retains every criticism and score disagreement.
The generator then revises. Stop after at most three rounds, an unchanged proposal, revision failure,
all-reviewer failure, or the request cap. All paper text is treated as untrusted data.


In [2]:
if RUN_LIVE:
    client = GeminiClient(settings, study / 'api_audit')
    run_study(seeds, client, retriever, settings, study, configurations=['full'])
else:
    print('Live specialist execution disabled; inspecting saved results.')


Live specialist execution disabled; inspecting saved results.


In [3]:
runs = [Run.model_validate(load_json(p)) for p in (study / 'runs').glob('*.json') if not p.name.endswith('.partial.json')]
print('Saved live runs:', sum(r.provenance == 'live_gemini' for r in runs))
if not runs:
    print('NOT RUN: No live generated proposals are available. Dataset and software checks are not experimental findings.')


Saved live runs: 12


In [4]:
for r in runs:
    if r.configuration != 'full':
        continue
    print('Seed:', r.seed_id, '| engine:', r.settings['resolved_engine'])
    print('Initial idea:', json.dumps(r.initial_idea.model_dump(), indent=2))
    for rd in r.rounds:
        print('Round', rd.index)
        for outcome in rd.reviews:
            print('Reviewer:', outcome.role, '| evidence status:', outcome.evidence_status)
            print(json.dumps(outcome.model_dump(), indent=2))
        print('Aggregation:', json.dumps(rd.aggregation, indent=2))
        print('Revision:', rd.revised_idea.model_dump() if rd.revised_idea else rd.revision_error)
    print('Final idea:', json.dumps(r.final_idea.model_dump(), indent=2))
    print('Stopping reason:', r.stop_reason)


Seed: 2025.naacl-long.342 | engine: python
Initial idea: {
  "title": "Integrating Automated Novelty Checking into Chain-of-Ideas Scientific Generation Frameworks",
  "research_problem": "How to dynamically filter and direct multi-step scientific idea generation paths using automated literature-grounded novelty assessment.",
  "research_gap": "Current chain-based and iterative LLM ideation frameworks effectively capture progressive domain developments and simulate peer reviews, but they lack real-time integration with rigorous, facet-based novelty checking mechanisms, often resulting in incremental ideas that overlap with existing literature.",
  "methodology": "We propose a modular pipeline combining Chain-of-Ideas (CoI) agent generation with an Idea Novelty Checker retrieval-augmented re-ranking module. During each step of the iterative idea evolution chain, candidate generation paths are queried against a vector database of scientific literature using a two-stage retrieve-then-reran

## Inspecting novelty verification
Search records preserve query, timestamp, provider, cache status, titles, authors, years, URLs and DOIs.
Evidence must cite a supplied ID and match an exact excerpt of the retrieved abstract.
This detects fabricated identifiers and quotations; it does not prove semantic entailment or universal
novelty. If search returns no usable evidence, the grounded novelty review is withheld and its failure
remains visible. The no-retrieval ablation retains seed context but removes additional novelty searches.

**Observed interaction examples:** Only saved live traces above count as actual agent interactions.
If none are present, this demonstration remains blocked on API/network access.
